<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/Lab3_Rideshare_Pricing_Instructor_EXECUTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — RideShare Pricing Engine: Instructor EXECUTED

Complete solution with example outputs.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
DATA_FILE = Path("lab3_rideshare_trips.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path(r"/mnt/data/module3_inheritance_polymorphism/lab3_rideshare_trips.csv")
print("Using:", DATA_FILE.resolve())

df = pd.read_csv(DATA_FILE)
df.head()

Using: /mnt/data/module3_inheritance_polymorphism/lab3_rideshare_trips.csv


,trip_id,vehicle_type,distance_km,duration_min,time_of_day,surge_multiplier,base_fare,per_km,per_min,total_fare_true
0,500001,Car,5.908,180.0,Midday,1.00,2.5,1.10,0.25,54.07
1,500002,Car,1.863,180.0,Night,1.10,2.5,1.10,0.25,50.22
2,500003,Car,2.299,180.0,AM_Peak,1.10,2.5,1.10,0.25,61.27
3,500004,Scooter,4.994,180.0,Midday,1.25,1.0,0.35,0.08,21.84
4,500005,XL,2.385,180.0,Midday,1.00,4.0,1.60,0.35,70.90


In [ ]:
print(df.shape)
df['vehicle_type'].value_counts()

(50000, 10)


vehicle_type
Car        22471
Scooter    12545
Bike        7541
XL          7443
Name: count, dtype: int64

In [ ]:
from abc import ABC, abstractmethod

TOD_MULT = {'AM_Peak':1.12,'Midday':1.00,'PM_Peak':1.18,'Night':0.92}

class Pricing(ABC):
    def __init__(self, base_fare, per_km, per_min):
        self.base_fare = float(base_fare)
        self.per_km = float(per_km)
        self.per_min = float(per_min)

    @abstractmethod
    def fare(self, distance_km, duration_min, surge_multiplier, time_of_day):
        ...

    def base_calc(self, distance_km, duration_min):
        return self.base_fare + self.per_km*distance_km + self.per_min*duration_min

class BikePricing(Pricing):
    def fare(self, distance_km, duration_min, surge_multiplier, time_of_day):
        return round(self.base_calc(distance_km, duration_min) * surge_multiplier * TOD_MULT[time_of_day], 2)

class ScooterPricing(Pricing):
    def fare(self, distance_km, duration_min, surge_multiplier, time_of_day):
        # scooters have a small safety fee
        return round((self.base_calc(distance_km, duration_min) + 0.25) * surge_multiplier * TOD_MULT[time_of_day], 2)

class CarPricing(Pricing):
    def fare(self, distance_km, duration_min, surge_multiplier, time_of_day):
        return round(self.base_calc(distance_km, duration_min) * surge_multiplier * TOD_MULT[time_of_day], 2)

class XLPricing(Pricing):
    def fare(self, distance_km, duration_min, surge_multiplier, time_of_day):
        # XL has a minimum fare
        fare = self.base_calc(distance_km, duration_min) * surge_multiplier * TOD_MULT[time_of_day]
        return round(max(6.0, fare), 2)


In [ ]:
def make_pricer(row):
    t = row['vehicle_type']
    cls = {'Bike':BikePricing,'Scooter':ScooterPricing,'Car':CarPricing,'XL':XLPricing}[t]
    return cls(row['base_fare'], row['per_km'], row['per_min'])


In [ ]:
sample = df.sample(8000, random_state=42)
pricers = [make_pricer(r) for _, r in sample.iterrows()]
pred = np.array([
    p.fare(r.distance_km, r.duration_min, r.surge_multiplier, r.time_of_day)
    for p, (_, r) in zip(pricers, sample.iterrows())
])
true = sample['total_fare_true'].to_numpy()
mae = np.mean(np.abs(true - pred))
mae

0.30071

In [ ]:
sample.assign(pred_fare=pred, abs_err=np.abs(true-pred)).groupby('vehicle_type')[['abs_err']].mean().sort_values('abs_err', ascending=False)

,abs_err
vehicle_type,
Scooter,0.369052
XL,0.281478
Car,0.275666
Bike,0.275239
